# Grok-rl-05-pg-ppo (hardened v2)

Episode-based REINFORCE / baseline / **PPO-clip** on CartPole from scratch.
PPO uses multi-epoch clipped updates on each episode batch (reliable on CartPole).


In [ ]:

import json, math, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.distributions import Categorical

OUT=Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
gpu={"cuda":torch.cuda.is_available(),"device_count":torch.cuda.device_count(),"names":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []}
print(gpu, device)

class CartPoleEnv:
    def __init__(self):
        self.g=9.8; self.mc=1.0; self.mp=0.1; self.tm=1.1; self.l=0.5; self.pml=0.05; self.f=10.0; self.tau=0.02
        self.th_lim=12*math.pi/180; self.x_lim=2.4
    def reset(self):
        self.state=np.random.uniform(-0.05,0.05,4).astype(np.float32); self.t=0; return self.state.copy()
    def step(self,a):
        x,xd,th,thd=map(float,self.state)
        force=self.f if a==1 else -self.f
        ct,st=math.cos(th),math.sin(th)
        temp=(force+self.pml*thd*thd*st)/self.tm
        thacc=(self.g*st-ct*temp)/(self.l*(4/3-self.mp*ct*ct/self.tm))
        xacc=temp-self.pml*thacc*ct/self.tm
        x+=self.tau*xd; xd+=self.tau*xacc; th+=self.tau*thd; thd+=self.tau*thacc
        self.state=np.array([x,xd,th,thd],np.float32); self.t+=1
        done=bool(abs(x)>self.x_lim or abs(th)>self.th_lim or self.t>=500)
        return self.state.copy(), (0.0 if done else 1.0), done, {}

class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared=nn.Sequential(nn.Linear(4,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh())
        self.pi=nn.Linear(64,2); self.v=nn.Linear(64,1)
    def forward(self,x):
        h=self.shared(x); return self.pi(h), self.v(h).squeeze(-1)


In [ ]:

def discount(rewards, gamma=0.99):
    G=0; out=[]
    for r in reversed(rewards):
        G=r+gamma*G; out.append(G)
    return list(reversed(out))

def run_reinforce(episodes=500, seed=0, baseline=False, lr=3e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    env=CartPoleEnv(); net=ActorCritic().to(device)
    opt=torch.optim.Adam(net.parameters(), lr=lr)
    hist=[]
    for _ in range(episodes):
        s=env.reset(); logps=[]; vals=[]; rewards=[]; done=False
        while not done:
            st=torch.tensor(s,device=device)
            logits,v=net(st); dist=Categorical(logits=logits); a=dist.sample()
            ns,r,done,_=env.step(int(a.item()))
            logps.append(dist.log_prob(a)); vals.append(v); rewards.append(r); s=ns
        Gs=torch.tensor(discount(rewards),device=device,dtype=torch.float32)
        logps=torch.stack(logps); vals=torch.stack(vals)
        if baseline:
            adv=Gs-vals.detach()
            loss=-(logps*adv).mean()+0.5*((vals-Gs)**2).mean()
        else:
            adv=(Gs-Gs.mean())/(Gs.std()+1e-8)
            loss=-(logps*adv).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        hist.append(sum(rewards))
    return np.array(hist)

def run_ppo(episodes=700, seed=0, clip=0.2, epochs=6, gamma=0.99, lr=3e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    env=CartPoleEnv(); net=ActorCritic().to(device)
    opt=torch.optim.Adam(net.parameters(), lr=lr)
    hist=[]
    for _ in range(episodes):
        s=env.reset(); states=[]; acts=[]; logps=[]; rewards=[]; vals=[]; done=False
        while not done:
            st=torch.tensor(s,device=device)
            with torch.no_grad():
                logits,v=net(st); dist=Categorical(logits=logits); a=dist.sample()
                logp=dist.log_prob(a)
            ns,r,done,_=env.step(int(a.item()))
            states.append(s); acts.append(int(a.item())); logps.append(float(logp)); vals.append(float(v)); rewards.append(r); s=ns
        Gs=np.array(discount(rewards,gamma),dtype=np.float32)
        adv=Gs-np.array(vals,dtype=np.float32)
        adv=(adv-adv.mean())/(adv.std()+1e-8)
        S=torch.tensor(np.array(states),device=device,dtype=torch.float32)
        A=torch.tensor(acts,device=device)
        old_logp=torch.tensor(logps,device=device)
        ADV=torch.tensor(adv,device=device); RET=torch.tensor(Gs,device=device)
        for _e in range(epochs):
            logits,v=net(S); dist=Categorical(logits=logits)
            logp=dist.log_prob(A)
            ratio=torch.exp(logp-old_logp)
            surr1=ratio*ADV; surr2=torch.clamp(ratio,1-clip,1+clip)*ADV
            loss=-torch.min(surr1,surr2).mean()+0.5*((v-RET)**2).mean()-0.01*dist.entropy().mean()
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(),0.5); opt.step()
        hist.append(sum(rewards))
    return np.array(hist)

t0=time.time()
ret_rf=run_reinforce(500, seed=1, baseline=False)
ret_bl=run_reinforce(500, seed=1, baseline=True)
ret_ppo=run_ppo(700, seed=1)
elapsed=time.time()-t0
print("RF",ret_rf[-50:].mean(),"BL",ret_bl[-50:].mean(),"PPO",ret_ppo[-50:].mean(),"elapsed",elapsed)


In [ ]:

def smooth(x,w=20):
    x=np.asarray(x,float)
    if len(x)<w: return x
    c=np.cumsum(np.insert(x,0,0)); return (c[w:]-c[:-w])/w
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(smooth(ret_rf),label="REINFORCE")
ax.plot(smooth(ret_bl),label="REINFORCE+baseline")
ax.plot(smooth(ret_ppo),label="PPO-clip")
ax.axhline(200,ls="--",c="gray",alpha=0.4)
ax.legend(); ax.set_title("Stage05 PG family (hardened)"); ax.set_xlabel("episode"); ax.set_ylabel("return")
fig.tight_layout(); fig.savefig(OUT/"stage05_pg_ppo.png",dpi=120); plt.close(fig)

payload={
  "ok": True, "stage":"05-pg-ppo", "title":"Grok-rl-05-pg-ppo",
  "metrics":{
    "reinforce_last50": float(ret_rf[-50:].mean()),
    "baseline_last50": float(ret_bl[-50:].mean()),
    "ppo_last50": float(ret_ppo[-50:].mean()),
    "ppo_max_avg50": float(max(ret_ppo[i:i+50].mean() for i in range(0,len(ret_ppo)-49))),
  },
  "gpu":gpu,"elapsed_sec":elapsed,
  "concept":"policy gradients; PPO clips probability ratio",
  "new_capability":"stable on-policy deep RL (PPO)",
  "compare_to_previous":"Stage04 Q-learning; Stage05 direct policy optimization",
  "fix_note":"episode-based PPO with multi-epoch clip updates",
}
assert payload["metrics"]["ppo_last50"] > 150, payload
assert payload["metrics"]["ppo_last50"] + 50 >= min(payload["metrics"]["reinforce_last50"], 300), payload
(OUT/"results_stage05.json").write_text(json.dumps(payload,indent=2))
print(json.dumps(payload,indent=2)); print("STAGE05_OK")
